# Day E4 — 데이터 정제: 단위, 이상치, 결측, 시간 정렬

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/E-ehr/day-E4.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/E-ehr/day-E4.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 단위 통일과 가능 범위 표시

In [ ]:
import numpy as np
import pandas as pd

# 예시 1: 단위 통일과 생리적 가능 범위
rng = np.random.default_rng(4)
n = 12
labs = pd.DataFrame({
    "stay_id": rng.integers(1, 5, n),
    "label": ["glucose"] * 6 + ["temperature"] * 6,
    "valuenum": [112, 6.1, 98, 7.4, 2300, 145, 37.2, 99.1, 36.8, 101.3, 3.7, 38.4],
    "valueuom": ["mg/dL", "mmol/L", "mg/dL", "mmol/L", "mg/dL", "mg/dL",
                 "°C", "°F", "°C", "°F", "°C", "°C"],
})
to_std = {("glucose", "mmol/L"): lambda v: v * 18.0,           # 포도당 mmol/L → mg/dL (근사 계수)
          ("temperature", "°F"): lambda v: (v - 32) * 5 / 9}   # 화씨 → 섭씨
std = labs.copy()
for (lab, unit), f in to_std.items():
    m = (std.label == lab) & (std.valueuom == unit)
    std.loc[m, "valuenum"] = f(std.loc[m, "valuenum"]).round(1)
    std.loc[m, "valueuom"] = "mg/dL" if lab == "glucose" else "°C"
plausible = {"glucose": (10, 1500), "temperature": (25, 45)}   # '불가능' 경계(가상 출발점)
std["plausible"] = [plausible[l][0] <= v <= plausible[l][1] for l, v in zip(std.label, std.valuenum)]
print(std.to_string(index=False))

### 예시 2 — 채혈 시각과 결과 입력 시각

In [ ]:
import pandas as pd
# 예시 2: 시간 정렬 -- '언제 알 수 있었는가'로 창을 자른다
stay = pd.DataFrame({"stay_id": [1], "intime": pd.to_datetime(["2150-03-01 08:00"])})
lact = pd.DataFrame({
    "stay_id": [1, 1, 1, 1],
    "charttime": pd.to_datetime(["2150-03-01 06:30", "2150-03-01 20:00",
                                 "2150-03-02 07:40", "2150-03-02 12:00"]),
    "storetime": pd.to_datetime(["2150-03-01 07:10", "2150-03-01 20:50",
                                 "2150-03-02 09:15", "2150-03-02 12:40"]),
    "lactate": [2.1, 3.4, 4.8, 2.0],
})
d = lact.merge(stay, on="stay_id")
d["cutoff"] = d.intime + pd.Timedelta(hours=24)               # 예측 시점: 입실 후 24시간
by_chart = d[(d.charttime >= d.intime - pd.Timedelta(hours=6)) & (d.charttime < d.cutoff)]
by_store = by_chart[by_chart.storetime < by_chart.cutoff]      # 그때 결과가 화면에 있었는가
print("\ncharttime 기준 최댓값:", by_chart.lactate.max(), "| storetime까지 고려한 최댓값:", by_store.lactate.max())

### 예시 3 — 결측은 무작위가 아니다, 그리고 가려서 채워 보기

In [ ]:
import numpy as np
import pandas as pd
# 예시 3: 결측은 무작위가 아니다 + 가려 놓고 채워 보기
rng = np.random.default_rng(13)
n = 3000
severity = rng.normal(0, 1, n)
age = rng.normal(65, 12, n)
lactate = np.exp(0.4 + 0.35 * severity + rng.normal(0, 0.25, n))
died = rng.random(n) < 1 / (1 + np.exp(-(-2.2 + 1.0 * severity)))
measured = rng.random(n) < 1 / (1 + np.exp(-(0.2 + 1.2 * severity)))   # 아플수록 잰다
df = pd.DataFrame({"age": age, "map": 75 - 6 * severity + rng.normal(0, 8, n),
                   "lactate": np.where(measured, lactate, np.nan), "died": died})
print("\n젖산을 잰 환자 사망률 %.3f, 안 잰 환자 사망률 %.3f" %
      (df.died[df.lactate.notna()].mean(), df.died[df.lactate.isna()].mean()))
print("잰 값의 평균 %.2f vs 모든 환자의 '진짜' 평균 %.2f" % (np.nanmean(df.lactate), lactate.mean()))

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
complete = df.dropna().reset_index(drop=True)[["age", "map", "lactate"]]
results = {}
for frac in [0.1, 0.3, 0.5]:
    for name, imp in [("평균", SimpleImputer(strategy="mean")),
                      ("중앙값", SimpleImputer(strategy="median")),
                      ("KNN(k=10)", KNNImputer(n_neighbors=10)),
                      ("회귀(반복)", IterativeImputer(random_state=0))]:
        errs = []
        for rep in range(5):
            r = np.random.default_rng(100 + rep)
            mask = r.random(len(complete)) < frac
            x = complete.copy(); x.loc[mask, "lactate"] = np.nan
            filled = imp.fit_transform(x)[:, 2]
            errs.append(np.sqrt(np.mean((filled[mask] - complete.lactate[mask]) ** 2)))
        results.setdefault(name, {})[f"{int(frac*100)}% 가림"] = round(float(np.mean(errs)), 3)
print("\n가린 젖산 값을 되살린 오차(RMSE, 작을수록 좋음)")
print(pd.DataFrame(results).T)